NAMED ENTITY RECOGNITION (NER)

Named Entity Recognition (NER) è una tecnica NLP che serve a individuare nel testo delle entità nominate e a classificarle per tipo

Named Entity Recognition è un'attività di NLP che individua nel testo segmenti corrispondenti a entità rilevanti e li classifica in categorie come persona, organizzazione, località, data o altre entità specifiche del dominio.

In pratica risponde a domande come:
come? dove? quale azienda? quale importo? quale organizzazione?
Esempio
"Leonardo Da Vinci ha dipinto la Gioconda a Firenze"
Un sistema NER potrebbe riconoscere
Leonardo Da Vinci -> persona
Firenze -> località
Quindi la pipeline è:
testo -> individuazione entità -> classificazione entità

Esempio:
"Apple ha aperto un nuovo ufficio a Milano nel 2025"
Apple -> organizzazione
Milano -> licalità
2025 -> data

La cosa importanto è che NER non sta classificando l'intera frase.
Questa è una distinzione fondamentale rispetto a quello che abbiamo visto con Spam Detection
"Vinci un iphone ora" -> spam  - stai assegnando una classa all'intero documento
"Microsoft apre una sede a Roma" -> Microsoft ORG - Roma LOC
Quindi CLASSIFICAZIONE TESTO -> assegna una classe all'intero documento
NER -> trova segmenti di testo e assegna una classe a ciascun segmento.

I tipi di entità dipendono dal modello, ma normalmente trovi:
PERSON/PER persone
ORG aziende, enti, organizzazioni
LOC/GPE città, stati, località
DATE date
TIME orari
MONEY importi
PRODUCT prodotti
EVENT eventi

Esempio: "Lisa ha incontrato Maria a New York il 15 giugno"
Lisa - PER
Maria - PER
New York - LOC
15 giugno - DATE
Il NER deve fare due cose:
- Capire dove inizia e dove finisce l'entità. Esempio New Yord, DNP Industriale SRL, devono essere trattati come singole entità. Questa fase è chiamata entity span detection
- Classificare l'entità

Ogni entità ha proprietà come:
- ent.text: testo
- ent.label_: il tipo, esempio PER per persona
- ent.start_char, ent.end_char: dove inizia e finisce l'entità
Avendo inizio e fine dell'entità puoi poi elaborare il testo in maniera strutturata.
Esempio, prendiamo una mail:
"Buongiorno, siamo Rossi srl e desideriamo ricevere 50 pezzi entro il 20 settembre"
Un distema NER potrebbe estrarre:
Rossi srl - ORG
500 - QUANTITA'
20 settembre - DATE
e trasformarlo in qualcosa del tipo:
{
 "cliente": "Rossi srl"
 "quantita": 500
 "data_consegna": 20 settembre   
}

Questo fa capire perchè NER è molto importante nell'automazione documentale.

Ma c'è una cosa importante: i modelli NER generici conoscono entità generiche.
Per esempio:
entità come: PERSON, DATE, ORG, LOG sono entità generiche
non necessariamente i modelli conoscono entità come:
CODICE_ARTICOLO, NUMERO_ORDINE, LOTTO, CODICE_CLIENTE, RIFERIMENTO_CLIENTE
Queste sono entità specifiche del DOMINIO
In un progetto aziendale potresti voler definire entità personalizzate:
CODICE_ARTICOLO -> pav1.1313.002
NUMERO_ORDINE -> ORD-45987
CLIENTE -> ABC Industriale SRL
DATA_CONSEGNA -> 30/09/2026

Qui entriamo nel CUSTOM NER, cioè NER addestrato sulle tue entità

NER generico e NER custom sono due problemi dello stesso tipo, ma con etichette differenti

Storicamente il NER è stato affrontato con metodi diversi. Oggi i Transformer sono molto usati perchè il conteto è fondamentale.
Esempio:
"Amazon ha assunto 500 persone" e "Ho navigato lungo l'Amazon"
"AmazoN" ha etichette diverse a seconda del contesto.
Il modello deve quindi osservare le parole vicine, questa è una delle ragioni per cui il NER moderno usa rappresentazioni contestuali

ATTENZIONE: NER non è infallibile, questo succede perhè il modello non usa un dizionario perfetto: fa una predizione probabilistica basata sul training.
NER quindi è una previsione statistica delle entità, non è un'estrazione perfetta.

Per questo in applicazioni aziendali i NER si combinano spesso con più strumenti:
NER + regex + dizionari aziendali + validazione database e+ eventuale LLM

NER è la capacità di una macchina non solo di leggere le parole, ma di distinguerne i protagonisti.
Mappa la realtà partendo dal testo grezzo.

Cosa significa riconoscere le entità?
differenza tra una parola comune ed un entità
Se dico mela sto parlando di un frutto, ma se dico Apple, in base al contesto, potrei riferirmi ad una delle aziende più grandi nel mondo.
Analizza il ruolo che quel sostantivo ricopre in una frase.
Prepara i dati pronti per un database.
Ma quali sono le etichette che usiamo per classififare il mondo?
Nel NLP standar lavoriamo con entità comuni. 
Il modello non tira ad indovinare, calcola una probabiltà condizionata, deve decidere se un termine appartiene a una classe 'C' data la sequenza di parole'x', massimizzando la probabilità condizionata.
Ogni parola influenza la classificazione della successiva, cercando di minimizzando l'incertezza.

Sfide di Riconoscimento
- Ambiguità semantica: una parola come 'Washington' può riferirsi a una persona, a una città, o a uno stato. Il NER deve risolvere l'ambiguità usando i token circostanti. Solo il contesto può dircelo.
- Entità Muilti-Token: le entità spesso sono composte da più parole (es. 'banca centrale europea' è una entità composta da 3 token). Il sistema deve identificare correttamente i confini dell'entità
- Variazioni Linguistiche: acronimi, abbreviazioni, e sinonimi richiedono modelli flessibili che non si basino solo su liste statiche di parole.

Per risolvere il problema dei confini tra le entità, usiamo un sistema di etichettatura tra le parole.

Tagging BIO
La codifica dei confini
Per gestire entità composte da più paroel, si usa lo schema BIO. 'B' (Begin) indica l'inizio dell'entità, 'I' (inside) la continuazione e 'O' (Outside) i token che non sono entità. 
Grazie a questa mappatura, trasformiamo  un problema di comprensione in un problema di classificazione
Questo trasforma i NET in un problema di classificazione per ogni singolo token.
Questo approccio permette di mappare la struttura sequenziale del linguaggio in modo che la rete possa imparare la dipendenza tra token consecutivi.

Ma chi fa effettivamente questi calcoli?
Entriamo nel cuore di spaCy
Come spaCy 'impara' a vedere le entità
A differenza dei vecchi sistemi basati su dizionari, spaCy utilizza modelli statistici pre-addestrati. 
Se usassimo solo liste non riconosceremo mai una startup appena nata o un nome straniero mai visto. 
Questi modelli non cercano la parola in un elenco, ma analizzano le caratteristiche morfologiche e sintattiche del testo per prevedere la probabilità che un termine sia un'entità
L'efficacia di spaCy deriva dall'uso di 'transition-based-systems' e reti neurali convoluzionali (CNN) o Transformer, a seconda della versione del modello caricata. Questo rende l'estrazioe estremamente veloce e precisa anche su testi mai visti prima.

Come scorrono i dati all'interno di questa architetura
- Quando caricate un modello spaCy, il componente 'EntityRecognizer' entra in azione dopo che il testo 
è già stato analizzato sintatticamente.
- Contextual Embeddings: utilizzo dei vettori delle parole circostanti per arricchire la rappresentazione del token corrente
- Transition Matrix: gestione degli stati interni per decidere se aprire, continuare o chiudere un'entità nominata.
- La perdita durante l'addestramento del modello statistico viene calcolata tramite la verosomiglianza logartimica negativa delle transazioni corrette.

In produzione la velocità spesso è tutto
Efficienza dei Modelli
Invece di avere una tabella gigante con milioni di parametri, usa la metematica per mappare le parole in uno spazio compresso.
- Modelli pre-addestrati: spaCy offre modelli ottimizzati per diverse lingue (come it_core_news_lg) che sono stati addestrati su milioni di frasi annotate.
- Hash Embeddings: per risparmiare memoria, spaCy usa una tecnica di hashing che permete di gestire vocabolari enormi senza esplosione dei parametri.
- Boom Filter: utilizzati internamento per verificare rapidamente l'appartenenza a classi note, migliorando la velocità di inferenza. per scartare istanteneamente ciò che sicuramente non è un'entità

Questo permette di far girare modelli enormi anche su macchine con risorse limitate.

E se il modello standard non conoscesse i termini del vostro settore specifico?

Aggiornamento del Modello.
Fine-tuning statistico
Nessuno modello è perfetto per ogni dominio.
I modelli di spaCy non sono statici. E' possibile eseguire il fine-tuning aggiungendo nuovi esempi di entità specifiche per un determinato dominio (es. legale o medico), permettendo al m odello di adattarsi a nuovi linguaggi.
L'aggiornamento avviene tramite discesa del gradiente, dove i pesi 'W' vengono modificati per ridurre l'errore di predizione sulle nuove entità.
Prendi il cervello pre-addestrato e lo sottoponi ad un aggiornamento intensivo sui nuovi dati.

Ora che abbiamo le entità come facciamo a mostrarle al cliente o al manager?
estrasrre i dati è utile ma vederli è un'altra cosa

Visualizzazione con displaCy
Rendere i dati leggibili agli umani.
Lavorare con liste di tuple o dizionari di Python è utile per il codice, ma per il debug e la presentazione dei risultati server una rappresentazione visiva.
Utilizzando lo strumento 'dispalcy' si possono evidenziare le entità direttamente nel testo, permettendo di verificare istantaneamente, la qualità dell'estrazione statistica eseguita dal modello.
Questo cambia come percepiamo la qualità del lavoro svolto.

Ma scopriamo le funzioni principali di questo strumento visuale
Funzionalità di Visualizzazione
Strumenti per l'ispezione delle entità
- Highlighting: evidenziazione cromatica differenziata per ogni tipo di entità (es. blu per le date, rosa per le persone, ecc)
- Filtering: possibilità di visualizzare solo specifiche categorie di entità per focalizzare l'analisi su dati critici
- Render Mode: supporto sia per la visualizzazione 'ent' (entità) che 'dep' (dipendenze sintattiche)
- La metrica di valutazioe dell'estrazione visiva si basa sulla previsione del riconoscimento dei confini dei token appartenenti all'entità

Possiamo portare queste grafiche fuori dal nostro codice?
E' possibile passare un dizionario di opzioni a 'displacy' per mappare colori personalizzati alle etichette, rendendo i report più leggibili.
L'argomento jupyter=True permette il rendering immediato all'interno delle celle dei notebook senza configurazioni aggiuntive
spaCy permete di esportare le visualizzazioni in formato statico per l'integrazione in documenti o siti web esterni (svg o html)

La visualizzazione ha un ruolo ancora più nobile, iol debug
Debug dele Entità
Individuare errori del modello.
La visualizzazione non è solo estetica, ma è un potente strumento di debug.
Vedendo come il modello sbaglia i confini o scambia una ORG per una PERSON, possiamo decidere se intervenire sul preprocessing o sul dataset di training.
L'errore di classificazione si verifica quando la classe predetta differisce dalla classe reale 'y' assegnata manualmente durante l'annotazione.

In [2]:
"""
================================================================================
FONDAMENTI DI NLP: NAMED ENTITY RECOGNITION (NER) CON SPACY
================================================================================
DESCRIZIONE ARCHITETTURALE:

1. INIZIALIZZAZIONE: Carichiamo il "modello linguistico", ovvero il cervello statistico (Classe Language).
2. ELABORAZIONE: Trasformiamo il testo grezzo (stringa) in un oggetto intelligente (Classe Doc).
3. ESTRAZIONE: Recuperiamo le entità nominate (nomi, luoghi, aziende) (Classe Span/Entity).
4. DECOSTRUZIONE: Analizziamo come il computer "vede" le entità tramite il sistema BIO (Classe Token).
5. VISUALIZZAZIONE: Creiamo un report grafico in HTML per rendere i dati leggibili agli umani.

INTERAZIONE TRA LE COMPONENTI:
- 'nlp' (Language): È il motore. Prende il testo e lo passa attraverso vari componenti (tokenizer, tagger, ner).
- 'doc' (Doc): È il risultato dell'elaborazione. Contiene il testo originale + tutti i metadati estratti.
- 'ent' (Span): Sono le "fette" del doc identificate come entità (es. "Elon Musk").
- 'token' (Token): Sono gli atomi del testo. Ogni singola parola o segno di punteggiatura.
================================================================================
"""

# Importiamo le librerie necessarie
import spacy                # La libreria principale per il Natural Language Processing
from spacy import displacy  # Sottogruppo di spacy dedicato alla visualizzazione grafica
import sys                  # Per gestire l'uscita forzata in caso di errore
import webbrowser           # Per aprire automaticamente il browser con i risultati
import os                   # Per gestire i percorsi dei file sul sistema

def carica_modello_nlp(nome_modello: str = "it_core_news_lg"):
    """
    Questa funzione prepara il 'motore' di intelligenza linguistica.
    """
    try:
        # Proviamo a caricare il modello specificato (di default quello 'large' per l'italiano)
        # nlp diventa un oggetto di classe 'Language' che contiene vocabolario e pesi neurali
        nlp = spacy.load(nome_modello)
        print(f"[SISTEMA] Modello '{nome_modello}' caricato e pronto all'uso.")
        return nlp
    except OSError:
        # Se il modello non è installato, diamo le istruzioni per scaricarlo
        print(f"[ERRORE] Il modello '{nome_modello}' non è stato trovato.")
        print("Esegui: 'python -m spacy download it_core_news_lg' nel tuo terminale.")
        sys.exit(1) # Esce dal programma perché non possiamo proseguire senza motore

def analizza_testo_ed_estrai_entita(nlp, testo: str):
    """
    Qui avviene la magia: trasformiamo una stringa di testo in dati strutturati.
    """
    # ESECUZIONE DELLA PIPELINE: 
    # nlp(testo) esegue in sequenza: tokenizzazione, tagging, parsing e infine NER.
    # L'oggetto 'doc' risultante è molto più di una stringa: sa TUTTO sulla grammatica e le entità.
    doc = nlp(testo)
    
    print("\n" + "="*60)
    print(" [REPORT] ENTITÀ IDENTIFICATE DAL MODELLO")
    print("="*60)
    
    # Controlliamo se sono state trovate entità (doc.ents contiene gli oggetti Span identificati)
    if not doc.ents:
        print("Nessuna entità trovata nel testo fornito.")
    else:
        # Cicliamo su ogni entità trovata nel documento
        for ent in doc.ents:
            # spacy.explain() ci dà una descrizione umana della categoria (es. 'PER' -> 'Person')
            spiegazione = spacy.explain(ent.label_)
            # Stampiamo il testo dell'entità, la sua etichetta tecnica e la spiegazione
            print(f"-> ENTITÀ: {ent.text:20} | CATEGORIA: {ent.label_:8} | DETTAGLIO: {spiegazione}")
    
    return doc # Restituiamo il doc elaborato per le fasi successive

def decodifica_logica_bio(doc):
    """
    Mostriamo come il computer 'etichetta' ogni singola parola internamente.
    NER usa il formato BIO: B=Begin (Inizio), I=Inside (Dentro), O=Outside (Fuori).
    """
    print("\n" + "="*60)
    print(" [DEBUG] ANALISI TECNICA: IL SISTEMA BIO")
    print("="*60)
    # Intestazione della tabella per la console
    print(f"{'PAROLA (TOKEN)':20} | {'TAG BIO':10} | {'SIGNIFICATO LOGICO'}")
    print("-" * 60)
    
    # Iteriamo su ogni singolo token (parola/punteggiatura) presente nel documento
    for token in doc:
        # token.ent_iob_ estrae il tag BIO (B, I oppure O)
        tag = token.ent_iob_
        
        # Traduciamo il tag in una descrizione comprensibile
        if tag == "B":
            stato = "INIZIO DI UN'ENTITÀ"   # Prima parola di un nome (es. 'Elon')
        elif tag == "I":
            stato = "PARTE DELL'ENTITÀ"    # Parola successiva (es. 'Musk')
        else:
            stato = "NON È UN'ENTITÀ"      # Parole comuni o punteggiatura
            
        # Stampiamo la riga corrispondente al token corrente
        print(f"{token.text:20} | {tag:10} | {stato}")

def genera_pagina_risultati(doc):
    """
    Crea una rappresentazione visiva (colorata) del testo e delle entità.
    """
    print("\n" + "="*60)
    print(" [VISUALIZZAZIONE] GENERAZIONE REPORT GRAFICO")
    print("="*60)
    
    # displacy.render trasforma il doc in codice HTML pronto per essere visualizzato
    # style="ent" indica che vogliamo evidenziare le entità nominate
    # page=True genera un documento HTML completo (col tag <html> e <body>)
    html_content = displacy.render(doc, style="ent", page=True, jupyter=False)
    
    # Nome del file dove salveremo il report
    file_name = "risultato_analisi_ner.html"
    
    # Salviamo la stringa HTML in un file fisico sul computer
    # 'utf-8' assicura che caratteri speciali (come lettere accentuate) siano salvati correttamente
    with open(file_name, "w", encoding="utf-8") as f:
        f.write(html_content)
    
    # Otteniamo il percorso completo del file appena creato
    abs_path = os.path.abspath(file_name)
    print(f"Report grafico generato con successo!")
    print(f"Puoi trovarlo qui: {abs_path}")
    
    # Comandiamo al sistema operativo di aprire il file HTML con il browser predefinito
    # Usiamo il prefisso 'file://' per indicare che è un file locale, non un sito web
    webbrowser.open(f"file://{abs_path}")

def main():
    """
    Questa è la funzione principale che orchestra l'intero processo.
    """
    # FASE 1: Preparazione - Carichiamo il cervello dell'IA
    # Usiamo 'it_core_news_lg' che è il modello più accurato per l'italiano
    nlp = carica_modello_nlp("it_core_news_lg")
    
    # FASE 2: Definizione del testo - Cosa vogliamo analizzare?
    testo_per_test = (
        "Mentre Apple annuncia nuovi uffici a Washington, "
        "Elon Musk vola in Italia per discutere con il Governo di Tesla. "
        "La Banca Centrale Europea osserva la situazione da Francoforte."
    )

    # FASE 3: Elaborazione - Passiamo il testo attraverso la pipeline di spaCy
    # Qui avviene il riconoscimento vero e proprio delle entità
    doc_elaborato = analizza_testo_ed_estrai_entita(nlp, testo_per_test)

    # FASE 4: Approfondimento - Vediamo i tag BIO dietro le quinte
    # Utile per capire come l'algoritmo separa i nomi comuni dalle entità
    decodifica_logica_bio(doc_elaborato)

    # FASE 5: Report Finale - Generiamo il file HTML interattivo
    # Apre automaticamente il browser per mostrare i risultati evidenziati
    genera_pagina_risultati(doc_elaborato)

# Questo blocco assicura che il codice parta solo se eseguiamo direttamente questo file
if __name__ == "__main__":
    main()

[SISTEMA] Modello 'it_core_news_lg' caricato e pronto all'uso.

 [REPORT] ENTITÀ IDENTIFICATE DAL MODELLO
-> ENTITÀ: Apple                | CATEGORIA: ORG      | DETTAGLIO: Companies, agencies, institutions, etc.
-> ENTITÀ: Washington           | CATEGORIA: LOC      | DETTAGLIO: Non-GPE locations, mountain ranges, bodies of water
-> ENTITÀ: Elon Musk            | CATEGORIA: PER      | DETTAGLIO: Named person or family.
-> ENTITÀ: Italia               | CATEGORIA: LOC      | DETTAGLIO: Non-GPE locations, mountain ranges, bodies of water
-> ENTITÀ: Governo di           | CATEGORIA: MISC     | DETTAGLIO: Miscellaneous entities, e.g. events, nationalities, products or works of art
-> ENTITÀ: Tesla                | CATEGORIA: PER      | DETTAGLIO: Named person or family.
-> ENTITÀ: Banca Centrale Europea | CATEGORIA: ORG      | DETTAGLIO: Companies, agencies, institutions, etc.
-> ENTITÀ: Francoforte          | CATEGORIA: LOC      | DETTAGLIO: Non-GPE locations, mountain ranges, bodies of w